# Programowanie w języku Python

## Wykład 8 — Pliki i serializacja danych

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć różnicę między danymi w pamięci a danymi trwałymi,
- korzystać z `pathlib.Path`,
- rozumieć ścieżki względne i bezwzględne,
- otwierać pliki w odpowiednim trybie,
- poprawnie używać `with`,
- czytać i zapisywać pliki tekstowe,
- rozumieć różnicę między tekstem i bajtami,
- korzystać z CSV,
- rozumieć pojęcia serializacji i deserializacji,
- zapisywać i odczytywać JSON,
- znać podstawowe zastosowanie `pickle`,
- umieć porównać rozmiar reprezentacji tekstowej i binarnej.

## Dane w pamięci a dane trwałe

Rozważmy:

```python
measurements = [21.7, 22.1, 20.8]
```

Lista istnieje podczas działania programu.

Po zakończeniu interpretera obiekt przestaje istnieć.

Jeżeli dane mają przetrwać zakończenie programu, musimy je zapisać.

## Pamięć operacyjna i plik

```text
PAMIĘĆ OPERACYJNA

zmienne
↓
obiekty Pythona
↓
dane istnieją podczas działania programu
```

```text
PLIK

dane zapisane na nośniku
↓
mogą istnieć po zakończeniu programu
```

Dzisiaj nauczymy się przechodzić między tymi dwoma światami.

## Ścieżka do pliku

Zanim otworzymy plik, musimy wiedzieć **gdzie on się znajduje**.

Współczesny Python udostępnia do tego klasę:

```python
from pathlib import Path
```

In [ ]:
from pathlib import Path

path = Path("data") / "measurements.txt"

print(path)
print(type(path))

## `Path` zamiast ręcznego składania napisów

Zamiast:

```python
"data/" + "measurements.txt"
```

możemy zapisać:

```python
Path("data") / "measurements.txt"
```

Operator `/` buduje ścieżkę w sposób czytelny i przenośny między systemami.

## Elementy ścieżki

Obiekt `Path` daje dostęp do podstawowych informacji:

In [ ]:
path = Path("data") / "measurements.csv"

print("name:", path.name)
print("stem:", path.stem)
print("suffix:", path.suffix)
print("parent:", path.parent)

## Ścieżka względna

```python
Path("data/measurements.txt")
```

jest ścieżką względną.

Jej znaczenie zależy od **bieżącego katalogu roboczego**.

In [ ]:
print(Path.cwd())

relative_path = Path("data") / "measurements.txt"

print(relative_path)
print(relative_path.absolute())

## Dlaczego pliku czasem „nie ma”?

Program może być uruchomiony z innego katalogu niż oczekujemy.

Dlatego przy problemach z plikami pierwsze pytanie brzmi:

```python
Path.cwd()
```

czyli:

> względem jakiego katalogu interpretowana jest ścieżka względna?

## Ścieżka bezwzględna

Ścieżka bezwzględna jednoznacznie wskazuje lokalizację w systemie plików.

```python
path.absolute()
```

Na co dzień często pracujemy jednak ze ścieżkami względnymi względem projektu.

## Sprawdzanie plików i katalogów

`Path` pozwala sprawdzić stan systemu plików:

In [ ]:
path = Path("data")

print(path.exists())
print(path.is_file())
print(path.is_dir())

## Tworzenie katalogu

```python
Path("data").mkdir()
```

zgłosi błąd, jeżeli katalog już istnieje.

Często wygodniej:

```python
Path("data").mkdir(exist_ok=True)
```

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

print(data_dir.exists())
print(data_dir.is_dir())

## Otwieranie pliku

Klasyczna funkcja:

```python
open(path, mode, encoding=...)
```

Przykład:

```python
open("data.txt", "r", encoding="utf-8")
```

zwraca obiekt reprezentujący otwarty plik.

## Najważniejsze tryby otwarcia

| Tryb | Znaczenie |
|---|---|
| `r` | odczyt |
| `w` | zapis, nadpisuje istniejący plik |
| `a` | dopisywanie na końcu |
| `x` | utworzenie nowego pliku |
| `b` | tryb binarny |

### Uwaga

`w` może usunąć dotychczasową zawartość pliku.

## Otwieranie i zamykanie ręczne

Możemy napisać:

```python
file = open("data.txt", "r", encoding="utf-8")
text = file.read()
file.close()
```

Problem:

> musimy pamiętać o zamknięciu pliku.

## `with` — poprawny wzorzec

Najczęściej używamy:

```python
with open("data.txt", "r", encoding="utf-8") as file:
    text = file.read()
```

Po wyjściu z bloku zasób zostaje poprawnie zwolniony.

## `Path.open()`

Obiekt `Path` również potrafi otworzyć plik:

```python
path = Path("data.txt")

with path.open("r", encoding="utf-8") as file:
    text = file.read()
```

W dalszej części wykładu będziemy często korzystać właśnie z tej postaci.

## Zapis tekstu

Najprostszy przykład:

In [ ]:
path = Path("data") / "example.txt"

with path.open("w", encoding="utf-8") as file:
    file.write("Pierwsza linia\n")
    file.write("Druga linia\n")

print(path.read_text(encoding="utf-8"))

## `w` a `a`

`w`:

```text
write
↓
tworzy plik lub zastępuje zawartość
```

`a`:

```text
append
↓
dopisuje na końcu istniejącego pliku
```

In [ ]:
path = Path("data") / "example.txt"

with path.open("a", encoding="utf-8") as file:
    file.write("Trzecia linia\n")

print(path.read_text(encoding="utf-8"))

## Czytanie całego pliku

```python
file.read()
```

zwraca cały tekst jako jeden obiekt `str`.

Dobre rozwiązanie dla małych plików.

Nie zawsze odpowiednie dla plików bardzo dużych.

In [ ]:
path = Path("data") / "example.txt"

with path.open("r", encoding="utf-8") as file:
    text = file.read()

print(text)
print(type(text))

## Czytanie jednej linii

```python
file.readline()
```

zwraca kolejną linię tekstu.

In [ ]:
with path.open("r", encoding="utf-8") as file:
    first = file.readline()
    second = file.readline()

print(repr(first))
print(repr(second))

## Najczęściej: iteracja po pliku

Plik możemy traktować jak obiekt iterowalny:

```python
with path.open(...) as file:
    for line in file:
        ...
```

Nie musimy wtedy wczytywać całego pliku do pamięci naraz.

In [ ]:
with path.open("r", encoding="utf-8") as file:
    for line in file:
        print(repr(line))

## Powrót do W6: przetwarzanie tekstu

Plik:

```text
sensor_01;temperature;21.7;C
sensor_02;temperature;22.4;C
sensor_03;humidity;45.2;%
```

możemy przetwarzać linię po linii.

In [ ]:
measurements_path = Path("data") / "measurements.txt"

measurements_path.write_text(
    "sensor_01;temperature;21.7;C\n"
    "sensor_02;temperature;22.4;C\n"
    "sensor_03;humidity;45.2;%\n",
    encoding="utf-8",
)

with measurements_path.open("r", encoding="utf-8") as file:
    for line in file:
        sensor, kind, value, unit = line.strip().split(";")
        print(sensor, kind, float(value), unit)

## Wygodne metody `Path`

Dla małych plików:

```python
path.read_text(encoding="utf-8")
path.write_text(text, encoding="utf-8")
```

są wygodnymi skrótami.

Ale:

> wczytują lub zapisują całą zawartość naraz.

## Plik tekstowy i kodowanie

Plik tekstowy fizycznie przechowuje **bajty**.

Aby zamienić je na `str`, Python musi znać kodowanie:

```python
encoding="utf-8"
```

To wraca bezpośrednio do W6.

## Tekst i bajty

```text
str
↓ encode("utf-8")
bytes
↓ zapis binarny
```

oraz:

```text
bytes
↓ decode("utf-8")
str
```

In [ ]:
text = "Zażółć"

data = text.encode("utf-8")

print(text, type(text))
print(data, type(data))

## Zapis binarny

Tryb `wb` oznacza zapis bajtów:

In [ ]:
binary_path = Path("data") / "message.bin"

data = "Zażółć".encode("utf-8")

with binary_path.open("wb") as file:
    file.write(data)

print(binary_path.read_bytes())

## Odczyt binarny

Tryb `rb` zwraca `bytes`:

In [ ]:
with binary_path.open("rb") as file:
    data = file.read()

text = data.decode("utf-8")

print(data, type(data))
print(text, type(text))

## CSV — dane tabelaryczne

Do prostych danych tabelarycznych często używamy CSV.

Przykład:

```text
sensor;type;value;unit
sensor_01;temperature;21.7;C
sensor_02;humidity;45.2;%
```

Nie zawsze warto parsować taki format ręcznie przez `split()`.

## Moduł `csv`

Python ma moduł standardowy:

```python
import csv
```

który poprawnie obsługuje m.in.:

- separatory,
- cudzysłowy,
- pola zawierające separator.

In [ ]:
import csv

csv_path = Path("data") / "measurements.csv"

csv_path.write_text(
    "sensor;type;value;unit\n"
    "sensor_01;temperature;21.7;C\n"
    "sensor_02;humidity;45.2;%\n",
    encoding="utf-8",
)

with csv_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file, delimiter=";")

    for row in reader:
        print(row)

## `DictReader`

Jeżeli pierwszy wiersz zawiera nazwy kolumn, wygodne jest:

```python
csv.DictReader
```

Każdy rekord otrzymujemy jako strukturę podobną do słownika.

In [ ]:
with csv_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file, delimiter=";")

    for row in reader:
        print(row)
        print(row["sensor"], row["value"])

## Zapis CSV przez `DictWriter`

Możemy również zapisywać słowniki jako rekordy:

In [ ]:
output_csv = Path("data") / "output.csv"

fieldnames = ["sensor", "type", "value", "unit"]

rows = [
    {
        "sensor": "sensor_01",
        "type": "temperature",
        "value": 21.7,
        "unit": "C",
    },
    {
        "sensor": "sensor_02",
        "type": "humidity",
        "value": 45.2,
        "unit": "%",
    },
]

with output_csv.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
        delimiter=";",
    )

    writer.writeheader()
    writer.writerows(rows)

print(output_csv.read_text(encoding="utf-8"))

## Serializacja

Teraz możemy formalnie zdefiniować pojęcie:

> **serializacja** to zamiana danych lub obiektów na reprezentację, którą można zapisać albo przesłać.

```text
obiekt Pythona
↓
serializacja
↓
tekst / bajty
↓
plik / sieć / baza
```

## Deserializacja

Operacja odwrotna:

```text
plik / sieć
↓
tekst / bajty
↓
deserializacja
↓
obiekty Pythona
```

Ważne:

> otrzymujemy nowe obiekty odtworzone na podstawie zapisanej reprezentacji.

## JSON

JSON jest popularnym tekstowym formatem wymiany danych.

Przykładowy obiekt Pythona:

In [ ]:
measurement = {
    "sensor": "sensor_01",
    "type": "temperature",
    "value": 21.7,
    "active": True,
}

print(measurement)
print(type(measurement))

## `json.dumps()`

```python
json.dumps(data)
```

serializuje obiekt Pythona do napisu JSON.

In [ ]:
import json

json_text = json.dumps(
    measurement,
    ensure_ascii=False,
)

print(json_text)
print(type(json_text))

## Python a JSON

| Python | JSON |
|---|---|
| `dict` | object |
| `list` | array |
| `str` | string |
| `int`, `float` | number |
| `True`, `False` | true, false |
| `None` | null |

Nie każdy dowolny obiekt Pythona można bezpośrednio zapisać jako JSON.

## Czytelny JSON

Możemy użyć:

```python
json.dumps(
    data,
    indent=2,
    ensure_ascii=False,
)
```

aby uzyskać reprezentację wygodniejszą dla człowieka.

In [ ]:
pretty_json = json.dumps(
    measurement,
    indent=2,
    ensure_ascii=False,
)

print(pretty_json)

## `dumps` a `dump`

```python
json.dumps(data)
```

zwraca napis.

```python
json.dump(data, file)
```

zapisuje JSON bezpośrednio do otwartego pliku.

Można zapamiętać:

```text
dumps → s jak string
dump  → plik
```

In [ ]:
json_path = Path("data") / "measurement.json"

with json_path.open("w", encoding="utf-8") as file:
    json.dump(
        measurement,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(json_path.read_text(encoding="utf-8"))

## `loads` a `load`

Analogicznie:

```python
json.loads(text)
```

deserializuje napis.

```python
json.load(file)
```

odczytuje JSON z otwartego pliku.

In [ ]:
with json_path.open("r", encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded)
print(type(loaded))

## JSON jako format wymiany danych

JSON nie jest formatem „tylko dla Pythona”.

```text
Python
  ↕
 JSON
  ↕
JavaScript / Java / C# / API / urządzenie
```

Dlatego jest bardzo popularny w komunikacji między systemami.

## Format tekstowy a binarny

Często słyszymy:

> „format binarny jest mniejszy”.

To **może być prawdą**, ale nie jest regułą gwarantowaną.

Sprawdźmy to eksperymentalnie na tych samych danych.

In [ ]:
measurements = [
    {
        "sensor": f"sensor_{i % 20:02d}",
        "type": "temperature",
        "value": round(20 + (i % 100) * 0.1, 1),
        "unit": "C",
        "active": True,
    }
    for i in range(1000)
]

len(measurements)

## JSON: wersja czytelna i kompaktowa

Najpierw dwie tekstowe reprezentacje:

```python
indent=2
```

oraz:

```python
separators=(",", ":")
```

Druga usuwa zbędne białe znaki.

In [ ]:
json_pretty = json.dumps(
    measurements,
    ensure_ascii=False,
    indent=2,
).encode("utf-8")

json_compact = json.dumps(
    measurements,
    ensure_ascii=False,
    separators=(",", ":"),
).encode("utf-8")

print(type(json_pretty))
print(type(json_compact))

## Format binarny: `pickle`

`pickle` serializuje obiekty Pythona do reprezentacji binarnej.

Na razie użyjemy go tylko do porównania rozmiaru.

In [ ]:
import pickle

pickle_data = pickle.dumps(
    measurements,
    protocol=pickle.HIGHEST_PROTOCOL,
)

print(type(pickle_data))

## Porównanie rozmiaru w bajtach

Ponieważ wszystkie trzy reprezentacje są teraz typu `bytes`, możemy bezpośrednio porównać ich długość.

In [ ]:
print(f"JSON czytelny:   {len(json_pretty):,} B")
print(f"JSON kompaktowy: {len(json_compact):,} B")
print(f"pickle:           {len(pickle_data):,} B")

## Porównanie względne

Możemy również policzyć stosunek rozmiarów:

In [ ]:
print(
    "JSON pretty / pickle:",
    round(len(json_pretty) / len(pickle_data), 2),
)

print(
    "JSON compact / pickle:",
    round(len(json_compact) / len(pickle_data), 2),
)

## Co właściwie mierzymy?

`len(...)` dla `bytes` podaje liczbę bajtów reprezentacji.

Czyli porównujemy:

```text
te same dane logiczne
↓
różne sposoby serializacji
↓
różna liczba bajtów
```

To jest porównanie formatu, a nie wielkości obiektów Pythona w RAM.

## Zapis na dysku

Możemy również zmierzyć rzeczywisty rozmiar zapisanych plików:

In [ ]:
pretty_path = Path("data") / "measurements_pretty.json"
compact_path = Path("data") / "measurements_compact.json"
pickle_path = Path("data") / "measurements.pkl"

pretty_path.write_bytes(json_pretty)
compact_path.write_bytes(json_compact)
pickle_path.write_bytes(pickle_data)

print(f"JSON pretty:  {pretty_path.stat().st_size:,} B")
print(f"JSON compact: {compact_path.stat().st_size:,} B")
print(f"pickle:       {pickle_path.stat().st_size:,} B")

## Dlaczego JSON może być większy?

JSON zapisuje jako tekst m.in.:

```json
{
  "sensor": "sensor_01",
  "value": 21.7,
  "unit": "C"
}
```

Nazwy pól, cyfry, przecinki, cudzysłowy i nawiasy również zajmują bajty.

W formacie binarnym dane mogą mieć inną, bardziej zwartą reprezentację.

## Ale „binarny = mniejszy” nie jest prawem

Rozmiar zależy od:

- rodzaju danych,
- powtarzalności wartości,
- sposobu kodowania,
- metadanych formatu,
- wersji formatu,
- zastosowanej kompresji,
- ustawień serializatora.

Dlatego:

> rozmiar mierzymy, a nie zakładamy.

## Czytelność kontra rozmiar

```text
JSON z indentacją
↓
łatwy dla człowieka
↓
większy
```

```text
JSON kompaktowy
↓
mniej czytelny
↓
mniejszy
```

```text
format binarny
↓
zwykle nieczytelny bez programu
↓
często bardziej zwarty
```

To jest przykład kompromisu projektowego.

## `pickle` — zapis i odczyt

Zapis:

In [ ]:
pickle_path = Path("data") / "measurement.pkl"

with pickle_path.open("wb") as file:
    pickle.dump(measurement, file)

## Odczyt `pickle`

```python
pickle.load(file)
```

odtwarza zapisany obiekt:

In [ ]:
with pickle_path.open("rb") as file:
    loaded_measurement = pickle.load(file)

print(loaded_measurement)
print(type(loaded_measurement))

## Bardzo ważne: bezpieczeństwo `pickle`

### Nigdy nie używaj `pickle.load()` dla danych z niezaufanego źródła.

Deserializacja `pickle` może prowadzić do wykonania kodu.

To oznacza, że plik `.pkl` powinien być traktowany bardziej jak program niż jak neutralny dokument.

## JSON a `pickle`

| Cecha | JSON | pickle |
|---|---|---|
| czytelny dla człowieka | tak | nie |
| między językami | tak | praktycznie nie |
| obsługa typów Pythona | ograniczona | znacznie szersza |
| format tekstowy | tak | nie |
| niezaufane dane | typowo znacznie bezpieczniejszy parser | **niebezpieczny** |
| zastosowanie | API, konfiguracja, wymiana danych | zaufane dane Pythona |

Dla początkującego:

> jeśli nie masz konkretnego powodu używać `pickle`, często lepiej zacząć od JSON.

## Jaki format wybrać?

| Format | Typowe zastosowanie |
|---|---|
| TXT | zwykły tekst |
| CSV | dane tabelaryczne |
| JSON | dane strukturalne, API, konfiguracja |
| plik binarny | surowe bajty |
| pickle | zaufana serializacja obiektów Pythona |

Nie istnieje jeden najlepszy format dla wszystkich problemów.

## Przykład integrujący

Mamy dane:

```python
measurements = [
    {
        "sensor": "sensor_01",
        "type": "temperature",
        "value": 21.7,
        "unit": "C",
    },
    ...
]
```

Chcemy zapisać je do pliku JSON, zakończyć program, a później je odtworzyć.

In [ ]:
measurements = [
    {
        "sensor": "sensor_01",
        "type": "temperature",
        "value": 21.7,
        "unit": "C",
    },
    {
        "sensor": "sensor_02",
        "type": "temperature",
        "value": 22.4,
        "unit": "C",
    },
]

path = Path("data") / "measurements.json"

path.parent.mkdir(exist_ok=True)

with path.open("w", encoding="utf-8") as file:
    json.dump(
        measurements,
        file,
        indent=2,
        ensure_ascii=False,
    )

In [ ]:
with path.open("r", encoding="utf-8") as file:
    loaded_measurements = json.load(file)

print(type(loaded_measurements))
print(type(loaded_measurements[0]))
print(loaded_measurements)

## Pełny cykl trwałości danych

```text
obiekty Pythona
↓
serializacja
↓
plik
↓
koniec programu
↓
plik
↓
deserializacja
↓
nowe obiekty Pythona
```

To jest najważniejszy schemat dzisiejszego wykładu.

## Co z błędami?

Przykład:

```python
open("missing.txt")
```

może zakończyć się:

```text
FileNotFoundError
```

Podobnie błędny JSON może zgłosić wyjątek.

Na tym wykładzie obserwujemy te sytuacje.

### Na następnym nauczymy się kontrolowanie na nie reagować.

## Program można rozumieć na wielu poziomach

```text
DANE
↓
dict / list / str / bytes

REPREZENTACJA
↓
CSV / JSON / pickle

SYSTEM PLIKÓW
↓
Path / plik / katalog

WYKONANIE
↓
read / write / with

KOSZT
↓
rozmiar pliku / zużycie pamięci / czytelność

BEZPIECZEŃSTWO
↓
zaufane i niezaufane dane
```

## Ćwiczenie

Napisz program, który:

1. tworzy katalog `data`,
2. zapisuje listę co najmniej 100 rekordów pomiarowych do JSON,
3. odczytuje zapisane dane,
4. zapisuje te same dane przy użyciu `pickle`,
5. mierzy rozmiary obu plików przez `Path.stat().st_size`,
6. wypisuje, który plik jest mniejszy i o ile procent,
7. wyjaśnia, dlaczego wynik nie jest uniwersalną regułą.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- `Path` reprezentuje ścieżkę w systemie plików,
- ścieżka względna zależy od katalogu roboczego,
- `with` jest podstawowym sposobem pracy z otwartym plikiem,
- `r`, `w`, `a` i `b` określają sposób pracy z plikiem,
- plik tekstowy wymaga kodowania,
- CSV jest wygodny dla danych tabelarycznych,
- serializacja zamienia dane na reprezentację możliwą do zapisania,
- JSON jest tekstowym i przenośnym formatem danych,
- `pickle` jest binarną serializacją specyficzną dla Pythona,
- `pickle` nie może być bezpiecznie używany do niezaufanych danych,
- format binarny bywa mniejszy od tekstowego, ale nie jest to gwarantowane,
- rzeczywisty rozmiar należy mierzyć.

## Następny wykład

# Błędy, wyjątki i diagnostyka programu

Po pracy z plikami naturalnie pojawiają się pytania:

- co zrobić, gdy pliku nie ma?
- co zrobić, gdy dane mają zły format?
- jak nie kończyć programu przy pierwszym błędzie?
- jak znaleźć źródło problemu?

To będzie temat Wykładu 9.